# SALITAyo Tagalog Open-Topic Restructurer V3

This notebook is built for your real goal: **open-topic Tagalog restructuring**, not just memorized templates.

It builds a broader dataset from public Filipino/Tagalog sources, creates simplified/active-voice targets, filters unsafe pairs, then trains your own mT5 candidate model.

What this notebook tries to teach:
- any Tagalog topic/structure -> clearer Tagalog
- same information, no hallucination
- active voice when possible
- hard/wordy terms -> common terms
- no bullets
- no copying when restructuring is possible

Important truth: there is no large public ready-made dataset for `open-topic Tagalog complex -> active/simple Tagalog`. So this notebook uses public Tagalog corpora as **source text** and builds target rewrites using strict rules and an optional teacher rewrite step. The teacher step is the closest to ?humanized? data without manually annotating thousands of rows.

## Setup
This version pins pandas for Colab and uses the modern `processing_class=tokenizer` trainer argument.

In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece scikit-learn requests
!pip -q install pandas==2.2.2

import os
import re
import json
import time
import random
import shutil
import requests
import pandas as pd
import numpy as np
import torch

from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('pandas:', pd.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Optional: Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/salitayo_tagalog_v3'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Drive output:', DRIVE_OUT)


## Configuration

For the best open-topic dataset, set `USE_TEACHER_REWRITE = True` and paste a Groq API key. If you skip the teacher step, the notebook still trains, but it will be less open-topic.

Teacher-generated data is only used to train **your own model**. Your app can still run locally after training.

In [ ]:
BASE_MODEL = 'google/mt5-base'
# BASE_MODEL = 'google/mt5-small'  # faster test run

TASK_PREFIX = 'restructure fil: '
OUTPUT_DIR = '/content/salitayo_tagalog_v3_training'
FINAL_DIR = '/content/salitayo_tagalog_v3_candidate_model'

MAX_INPUT_LENGTH = 224
MAX_TARGET_LENGTH = 224
EPOCHS = 4
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-5

# Open-topic source collection limits
MAX_SOURCE_SENTENCES = 9000
MAX_TEACHER_PAIRS = 1800   # increase to 3000-5000 if you have time/API budget

# Set True if you can use Groq to create humanized target rewrites.
USE_TEACHER_REWRITE = False
GROQ_API_KEY = ''  # paste key only in your private Colab if using teacher rewrite
GROQ_MODEL = 'llama-3.1-8b-instant'

USER_DATA_PATH = ''  # optional CSV/jsonl with original,restructured or complex,simple


## Lexical Dictionary and Safety Helpers

This dictionary is not the final behavior. It is used for controlled examples and safety filtering.

In [ ]:
HARD_TO_COMMON = {
    'akumulasyon': 'pag-ipon', 'analisis': 'pagsusuri', 'asistensya': 'tulong',
    'benepisyo': 'pakinabang', 'departamento': 'tanggapan', 'detalyado': 'may detalye',
    'epektibo': 'mabisa', 'ekstensibo': 'malawak', 'eliminahin': 'alisin',
    'emosyonal': 'damdamin', 'estratehiya': 'paraan', 'evaluasyon': 'pagsusuri',
    'identipikasyon': 'pagkilala', 'implementasyon': 'pagsasagawa', 'impormasyon': 'kaalaman',
    'inisyal': 'una', 'interaksyon': 'pakikipag-ugnayan', 'interpretasyon': 'pag-unawa',
    'isinasagawa': 'ginagawa', 'isinagawa': 'ginawa', 'kinakailangan': 'kailangan',
    'kognitibo': 'pag-iisip', 'kolaborasyon': 'pagtutulungan', 'komplikado': 'mahirap',
    'komplikadong': 'mahirap na', 'komprehensibo': 'kumpleto', 'komunikasyon': 'pakikipag-usap',
    'konklusyon': 'wakas', 'konsekwensya': 'bunga', 'konsiderable': 'malaki',
    'konstruksyon': 'pagtatayo', 'kontemporaryo': 'makabago', 'konteksto': 'kalagayan',
    'makabuluhan': 'mahalaga', 'malawakang': 'malawak', 'maramihang': 'marami',
    'metodolohiya': 'paraan', 'obserbasyon': 'pagmamasid', 'organisasyon': 'samahan',
    'partisipasyon': 'pagsali', 'perspektibo': 'pananaw', 'potensyal': 'posible',
    'prayoridad': 'una', 'presentasyon': 'paglalahad', 'proseso': 'hakbang',
    'proyekto': 'gawain', 'rekomendasyon': 'mungkahi', 'representasyon': 'paglalarawan',
    'responsibilidad': 'tungkulin', 'resulta': 'kinalabasan', 'signipikante': 'mahalaga',
    'sitwasyon': 'kalagayan', 'solusyon': 'lunas', 'teknolohiya': 'makinarya',
    'terminolohiya': 'mga salita', 'asenso': 'pag-unlad', 'suliranin': 'problema',
    'kumplikado': 'mahirap', 'makabagong': 'bagong', 'kakayahan': 'abilidad',
    'digital': 'pangkompyuter', 'pormal': 'opisyal', 'antas': 'lebel',
    'responde': 'sumagot', 'rumesponde': 'sumagot', 'kagamitan': 'gamit',
    'katuwang': 'kasama', 'pag-unlad': 'paglago',
}

PROTECTED_PATTERN = re.compile(r'\b(?:[A-Z]{2,}|[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+|\d+(?:\.\d+)?%?)\b')
TAGALOG_MARKERS = {'ang','ng','mga','sa','ay','na','si','sina','nila','natin','ito','iyan','iyon','para','hindi','kung','may'}


def normalize_spaces(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def split_sentences(text):
    text = normalize_spaces(text)
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if 4 <= len(p.split()) <= 35]


def looks_filipino(text):
    words = set(re.findall(r"[A-Za-z?-?']+", str(text).lower()))
    return len(words & TAGALOG_MARKERS) >= 1


def protected_terms(text):
    return set(m.group(0) for m in PROTECTED_PATTERN.finditer(str(text)))


def normalize_words(text):
    return re.findall(r"[A-Za-z?-?0-9'\-]+", str(text).lower())


def match_case(source, replacement):
    if source.isupper(): return replacement.upper()
    if source[:1].isupper(): return replacement[:1].upper() + replacement[1:]
    return replacement


def lexical_replace_tl(text):
    protected = [m.span() for m in PROTECTED_PATTERN.finditer(text)]
    def blocked(start, end):
        return any(start < e and end > s for s, e in protected)
    replacements = []
    for hard, common in sorted(HARD_TO_COMMON.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = re.compile(rf'\b{re.escape(hard)}\b', flags=re.IGNORECASE)
        for m in pattern.finditer(text):
            if not blocked(m.start(), m.end()):
                replacements.append((m.start(), m.end(), match_case(m.group(0), common), m.group(0)))
    out = text
    used = []
    occupied = []
    for start, end, common, original in sorted(replacements, reverse=True):
        if any(start < e and end > s for s, e in occupied):
            continue
        out = out[:start] + common + out[end:]
        used.append((original, common))
        occupied.append((start, end))
    return out, used


def pair_is_safe(original, target):
    original = normalize_spaces(original)
    target = normalize_spaces(target)
    if not original or not target or original.lower() == target.lower(): return False
    if '?' in target or '\n' in target: return False
    if not looks_filipino(target): return False
    o_words = normalize_words(original)
    t_words = normalize_words(target)
    if not o_words or not t_words: return False
    ratio = len(t_words) / max(len(o_words), 1)
    if ratio < 0.45 or ratio > 1.55: return False
    for term in protected_terms(original):
        if term not in target: return False
    return True


## Collect Open-Topic Filipino Source Sentences

Sources:
- `saillab/alpaca_filipino_taco`: broad instruction/response Filipino text
- `UD-Filipino/UD_Tagalog-NewsCrawl`: Tagalog news crawl sentences
- `jfernandez/cebuano-filipino-sentences`: Filipino side of sentence pairs
- `data-is-better-together/MPEP_FILIPINO`: Filipino target/suggestions
- `josephimperial/filipino_elementary_periodical_exams`: elementary Filipino text


In [ ]:
def extract_response_filipino(text):
    text = str(text or '')
    m = re.search(r'Response in Filipino:\s*(.*)', text, flags=re.S)
    if m:
        return re.sub(r'<\|.*?\|>|Response in English:.*', '', m.group(1), flags=re.S).strip()
    return text.strip()


def add_sentence(sentences, text):
    for s in split_sentences(text):
        if looks_filipino(s):
            sentences.append(s)


def collect_sources(max_sentences=MAX_SOURCE_SENTENCES):
    sentences = []

    # Broad instruction-response Filipino text
    try:
        ds = load_dataset('saillab/alpaca_filipino_taco', split='train')
        for row in ds.select(range(min(len(ds), 7000))):
            add_sentence(sentences, row.get('instruction', ''))
            add_sentence(sentences, extract_response_filipino(row.get('output', '')))
            if len(sentences) >= max_sentences: break
    except Exception as exc:
        print('Skipping alpaca_filipino_taco:', exc)

    # Tagalog NewsCrawl
    try:
        ds = load_dataset('UD-Filipino/UD_Tagalog-NewsCrawl', split='train')
        for row in ds.select(range(min(len(ds), 6000))):
            add_sentence(sentences, row.get('text', ''))
            if len(sentences) >= max_sentences: break
    except Exception as exc:
        print('Skipping UD Tagalog:', exc)

    # Filipino side from Cebuano-Filipino pairs
    try:
        ds = load_dataset('jfernandez/cebuano-filipino-sentences', split='train')
        for row in ds.select(range(min(len(ds), 9000))):
            pair = row.get('set', [])
            if isinstance(pair, list) and pair:
                add_sentence(sentences, pair[0])
            if len(sentences) >= max_sentences: break
    except Exception as exc:
        print('Skipping cebuano-filipino:', exc)

    # MPEP Filipino targets/suggestions
    try:
        ds = load_dataset('data-is-better-together/MPEP_FILIPINO', split='train')
        for row in ds:
            for item in row.get('target', []) or []:
                if isinstance(item, dict): add_sentence(sentences, item.get('value', ''))
            add_sentence(sentences, row.get('target-suggestion', ''))
    except Exception as exc:
        print('Skipping MPEP:', exc)

    # Elementary exam questions/options
    try:
        ds = load_dataset('josephimperial/filipino_elementary_periodical_exams', split='train')
        for row in ds:
            add_sentence(sentences, row.get('question', ''))
            for opt in row.get('options', []) or []:
                add_sentence(sentences, opt)
    except Exception as exc:
        print('Skipping elementary exams:', exc)

    clean, seen = [], set()
    for s in sentences:
        s = normalize_spaces(s)
        if s and s not in seen and 4 <= len(s.split()) <= 35:
            clean.append(s); seen.add(s)
    random.shuffle(clean)
    return clean[:max_sentences]

source_sentences = collect_sources()
print('Source sentences:', len(source_sentences))
for s in source_sentences[:15]: print('-', s)


## Rule-Based Pair Builder
This gives safe broad pairs without API use. It is not enough by itself for perfect open-topic behavior, but it gives coverage across real sentences.

In [ ]:
def restructure_ay_sentence(text):
    s = normalize_spaces(text).rstrip('.!?')
    m = re.match(r'^Ang\s+(.+?)\s+ay\s+(.+)$', s, flags=re.I)
    if not m:
        return None
    subject = m.group(1).strip()
    predicate = m.group(2).strip()
    if len(subject.split()) > 12 or len(predicate.split()) > 14:
        return None
    # avoid weird question forms
    if predicate.lower().startswith(('sino','ano','bakit','paano','saan','kailan')):
        return None
    return f'{predicate[:1].upper() + predicate[1:]} ang {subject}.'


def passive_to_active_tl(text):
    s = normalize_spaces(text).rstrip('.!?')
    m = re.match(r'^Ang\s+(.+?)\s+ay\s+(.+?)\s+ng\s+(.+)$', s, flags=re.I)
    if not m: return None
    obj, verb, actor = [x.strip() for x in m.groups()]
    if len(obj.split()) > 10 or len(actor.split()) > 10: return None
    return f'{verb[:1].upper() + verb[1:]} ng {actor} ang {obj}.'


def build_rule_pairs(sentences):
    pairs=[]; seen=set()
    for s in sentences:
        candidates=[]
        lex, used = lexical_replace_tl(s)
        if used: candidates.append(lex)
        active = passive_to_active_tl(s)
        if active: candidates.append(active)
        ay = restructure_ay_sentence(s)
        if ay: candidates.append(ay)
        # combine lexical + ay/active if possible
        if used:
            active2 = passive_to_active_tl(lex) or restructure_ay_sentence(lex)
            if active2: candidates.append(active2)
        for t in candidates:
            if pair_is_safe(s,t):
                key=(s,t)
                if key not in seen:
                    pairs.append({'original':s,'restructured':t,'source':'open_topic_rules'})
                    seen.add(key)
    return pairs

rule_pairs = build_rule_pairs(source_sentences)
print('Rule pairs:', len(rule_pairs))
for row in rule_pairs[:8]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Optional Teacher Rewrite: Humanized Open-Topic Targets

This is the key part if you want broader generalization. It asks a stronger model to rewrite real Tagalog sentences into simpler active Tagalog, then filters unsafe outputs.

If you do not have an API key, skip this. The model will still train, but it will be less open-topic.


In [ ]:
def teacher_rewrite(sentence):
    if not USE_TEACHER_REWRITE or not GROQ_API_KEY:
        return None
    prompt = f"""Gawain: Isulat muli ang pangungusap sa mas malinaw, mas karaniwang Tagalog.
Panuntunan:
- Panatilihin ang parehong impormasyon.
- Huwag magdagdag ng bagong detalye.
- Huwag magtanggal ng pangalan, lugar, numero, petsa, o mahalagang paksa.
- Gawing active voice kung natural.
- Palitan ang mahirap na salita ng mas karaniwang salita kung maaari.
- Isang pangungusap lamang ang ibalik.
- Walang paliwanag, walang bullet.

Pangungusap: {sentence}
Sagot:"""
    try:
        r = requests.post(
            'https://api.groq.com/openai/v1/chat/completions',
            headers={'Authorization': f'Bearer {GROQ_API_KEY}', 'Content-Type': 'application/json'},
            json={
                'model': GROQ_MODEL,
                'messages': [{'role': 'user', 'content': prompt}],
                'temperature': 0.1,
                'max_tokens': 180,
            },
            timeout=20,
        )
        r.raise_for_status()
        out = r.json()['choices'][0]['message']['content'].strip()
        out = re.sub(r'^(Sagot:|Output:|Result:)\s*', '', out, flags=re.I).strip().strip('"')
        return normalize_spaces(out)
    except Exception as exc:
        print('teacher error:', exc)
        return None

teacher_pairs=[]
if USE_TEACHER_REWRITE and GROQ_API_KEY:
    for i, s in enumerate(source_sentences[:MAX_TEACHER_PAIRS], 1):
        t = teacher_rewrite(s)
        if t and pair_is_safe(s,t):
            teacher_pairs.append({'original':s,'restructured':t,'source':'teacher_humanized'})
        if i % 50 == 0:
            print('teacher processed', i, 'kept', len(teacher_pairs))
            time.sleep(1)
else:
    print('Teacher rewrite skipped. Set USE_TEACHER_REWRITE=True and GROQ_API_KEY to create humanized targets.')

print('Teacher pairs:', len(teacher_pairs))
for row in teacher_pairs[:8]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Controlled Active Voice + Lexical Coverage
These pairs keep the model grounded in the exact active-voice and hard-word behavior you want.

In [ ]:
ACTORS = ['guro','mag-aaral','mga mag-aaral','mananaliksik','mga mananaliksik','doktor','tagapagsalita','pangkat','komunidad','paaralan','SALITAyo']
OBJECTS = ['datos','ulat','aralin','impormasyon','proseso','proyekto','gawain','tanong','sagot','plano','resulta','dokumento','kuwento','pananaliksik']
VERBS = [('sinuri','Sinuri'),('ipinaliwanag','Ipinaliwanag'),('isinulat','Isinulat'),('binasa','Binasa'),('ginawa','Ginawa'),('sinimulan','Sinimulan'),('tinapos','Tinapos'),('ipinakita','Ipinakita'),('inayos','Inayos'),('tinulungan','Tinulungan')]

controlled=[]; seen=set()
def add_pair(o,t,source):
    if pair_is_safe(o,t) and (o,t) not in seen:
        controlled.append({'original':o,'restructured':t,'source':source}); seen.add((o,t))

for actor in ACTORS:
    for obj in OBJECTS:
        for passive, active in VERBS:
            add_pair(f'Ang {obj} ay {passive} ng {actor}.', f'{active} ng {actor} ang {obj}.', 'active_voice_template')

subjects = ['asenso','tagumpay','pagkatuto','komunikasyon','partisipasyon','implementasyon','edukasyon','pananaliksik','proyekto','proseso','solusyon','rekomendasyon','impormasyon','metodolohiya','aralin','ulat','gawain','paliwanag','panuto','desisyon','plano','layunin']
predicates = ['mahalaga','malinaw','kailangan','makabuluhan','mahirap unawain','madaling sundin','kapaki-pakinabang','walang nakikitang sulok','hindi laging madali','hindi agad nakikita','hindi dapat minamadali']
for subj in subjects:
    for pred in predicates:
        o=f'Ang {subj} ay {pred}.'
        lex, _ = lexical_replace_tl(o)
        t=restructure_ay_sentence(lex) or lex
        add_pair(o,t,'ay_restructure_template')

for hard in HARD_TO_COMMON:
    for obj in OBJECTS[:8]:
        o=f'Ang {hard} ay mahalaga sa {obj}.'
        t,_=lexical_replace_tl(o)
        t=restructure_ay_sentence(t) or t
        add_pair(o,t,'lexical_template')

print('Controlled pairs:', len(controlled))
for row in controlled[:8]:
    print('\nORIGINAL:', row['original'])
    print('TARGET  :', row['restructured'])


## Optional User Pairs

In [ ]:
def load_user_pairs(path):
    if not path: return []
    if not os.path.exists(path): raise FileNotFoundError(path)
    if path.endswith('.csv'): df = pd.read_csv(path)
    elif path.endswith('.jsonl'): df = pd.read_json(path, lines=True)
    elif path.endswith('.json'): df = pd.read_json(path)
    else: raise ValueError('Use CSV, JSON, or JSONL')
    if {'complex','simple'}.issubset(df.columns):
        df=df.rename(columns={'complex':'original','simple':'restructured'})
    if not {'original','restructured'}.issubset(df.columns):
        raise ValueError('Need original/restructured or complex/simple columns')
    rows=[]
    for _,r in df[['original','restructured']].dropna().iterrows():
        o,t=normalize_spaces(r['original']), normalize_spaces(r['restructured'])
        if pair_is_safe(o,t): rows.append({'original':o,'restructured':t,'source':'user_checked'})
    return rows

user_pairs = load_user_pairs(USER_DATA_PATH)
print('User pairs:', len(user_pairs))


## Build Final Dataset

Best quality mix:
- teacher_humanized if available
- open_topic_rules from real corpora
- controlled active/lexical examples
- your checked pairs if provided

In [ ]:
all_pairs = teacher_pairs + rule_pairs + controlled + user_pairs
df = pd.DataFrame(all_pairs)[['original','restructured','source']].drop_duplicates().reset_index(drop=True)
before=len(df)
df = df[df.apply(lambda r: pair_is_safe(r['original'], r['restructured']), axis=1)].reset_index(drop=True)
print('Before filter:', before)
print('After filter:', len(df))
print(df['source'].value_counts())

if len(df) < 3000:
    print('WARNING: Dataset is small. For open-topic behavior, use teacher rewrite or add checked pairs.')

train_df, eval_df = train_test_split(df, test_size=0.08, random_state=SEED, stratify=df['source'] if df['source'].nunique() > 1 else None)
print('Train:', len(train_df), 'Eval:', len(eval_df))
df.sample(min(10,len(df)), random_state=SEED)


## Train mT5 Candidate

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))

def preprocess(batch):
    inputs = [TASK_PREFIX + x for x in batch['original']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch['restructured'], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs['labels'] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
eval_tok = eval_ds.map(preprocess, batched=True, remove_columns=eval_ds.column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=False,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
    data_collator=collator,
)
trainer.train()


## Open-Topic Acceptance Tests

In [ ]:
def simplify_tl(text, max_new_tokens=180):
    prompt = TASK_PREFIX + text.strip()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_INPUT_LENGTH).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=0.95,
            early_stopping=True,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

GOLD_TESTS = [
    'Ang tunay na asenso ay walang iniikitang sulok.',
    'Ngayong araw, pormal nating binubuksan ang pinto sa susunod na antas ng digital na pakikipagtulungan.',
    "Sa pamamagitan ng mga makabagong sistema, umuunawa tayo ng iba't ibang wika at rumesponde sa kumplikadong suliranin.",
    'Ang mahirap na gawain ay tinapos ng mga mag-aaral bago maghapon.',
    'Ang detalyadong impormasyon ay ibinigay ng doktor sa pasyente.',
    'Ang proyekto ay sinimulan ng pangkat noong Lunes.',
    'Ang komplikadong panuto ay binasa ng estudyante sa silid-aralan.',
    'Ang rekomendasyon para sa paaralan ay isinulat ng komunidad.',
]

for text in GOLD_TESTS:
    pred = simplify_tl(text)
    print('\nINPUT :', text)
    print('OUTPUT:', pred)
    print('COPY? :', pred.strip().lower() == text.strip().lower())
    print('BULLET:', '?' in pred)


## Save Candidate Model

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
with open(os.path.join(FINAL_DIR, 'salitayo_tagalog_v3_config.json'), 'w') as f:
    json.dump({'task':'open_topic_tagalog_restructure','base_model':BASE_MODEL,'task_prefix':TASK_PREFIX,'pairs':len(df),'sources':df['source'].value_counts().to_dict()}, f, indent=2)

if 'DRIVE_OUT' in globals():
    drive_model_dir = os.path.join(DRIVE_OUT, 'salitayo_tagalog_v3_candidate_model')
    if os.path.exists(drive_model_dir): shutil.rmtree(drive_model_dir)
    shutil.copytree(FINAL_DIR, drive_model_dir)
    print('Saved to Drive:', drive_model_dir)

shutil.make_archive('/content/salitayo_tagalog_v3_candidate_model', 'zip', FINAL_DIR)
print('Candidate model:', FINAL_DIR)
print('Zip:', '/content/salitayo_tagalog_v3_candidate_model.zip')


## What To Expect

If `USE_TEACHER_REWRITE=False`, this is better than the template-only model but still not perfect open-topic Tagalog.

For the strongest result, use `USE_TEACHER_REWRITE=True` for 1,800+ teacher pairs. That is the closest practical way to create humanized open-topic targets without manually writing thousands of examples.

Expected training time on T4:
- `mt5-small`: about 1 to 2 hours
- `mt5-base`: about 2.5 to 5 hours, depending on dataset size
